## Real-Time Model Serving

### Purpose

This notebook deploys the approved `Champion` model to a real-time serving endpoint.

It creates or updates a Databricks Model Serving endpoint so the model can be invoked via REST API for low-latency inference.

This is the final step in Stage 3 of the BYOM workflow (Evaluate → Approve → Deploy).

### Preconditions

Before running this notebook:

- The model must be registered in Unity Catalog
- The `Champion` alias must be assigned (via approval stage)
- Evaluation and approval must be complete

This notebook assumes governance has already been enforced.

### What This Notebook Does

- Loads the `Champion` model reference
- Creates or updates a serving endpoint
- Configures compute scaling
- Deploys the model to the endpoint
- Validates endpoint availability

The endpoint always references the `Champion` alias, not a specific version number.


In [0]:
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

### Configure Endpoint Parameters

Define:

- `model_name`
- `endpoint_name`
- Scaling configuration (min/max replicas)
- Compute size (if applicable)

Align configuration with expected traffic patterns and latency requirements.


In [0]:
# Widget parameters for job orchestration
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("model_name", "pyfunc_xgb", "Model Name")  # same as model_type when logging, or dl model name from artifacts.json
dbutils.widgets.text("model_version", "", "Model Version (leave empty for Champion)")
dbutils.widgets.dropdown("workload_size", "Small", ["Small", "Medium", "Large"], "Workload Size")
dbutils.widgets.dropdown("scale_to_zero", "true", ["true", "false"], "Scale to Zero")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
model_name = dbutils.widgets.get("model_name")
model_version = dbutils.widgets.get("model_version")
workload_size = dbutils.widgets.get("workload_size")
scale_to_zero = dbutils.widgets.get("scale_to_zero") == "true"

# Construct registered model name
registered_model_name = f"{catalog}.{schema}.{model_name}"
endpoint_name = f"{model_name}-endpoint"

### Create or Update Serving Endpoint

If the endpoint does not exist:
- Create a new endpoint.

If the endpoint exists:
- Update it to reference the current `Champion`.

This ensures:
- Controlled promotion
- Zero manual version switching
- Clear rollback path (by reassigning alias)

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
from databricks.sdk.errors import NotFound
from datetime import datetime
import time

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
w = WorkspaceClient()

# Resolve Champion version if not specified
if not model_version:
    champion_mv = client.get_model_version_by_alias(registered_model_name, "Champion")
    model_version = champion_mv.version
model_version = int(model_version)

# Verify model is approved
mv = client.get_model_version(registered_model_name, str(model_version))
if mv.tags.get("approval") != "approved":
    raise ValueError(f"Model version {model_version} is not approved. Run 3_model_approval first.")

# Create or update serving endpoint
served_entity = ServedEntityInput(
    name=f"{model_name}-v{model_version}",
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_size=workload_size,
    scale_to_zero_enabled=scale_to_zero,
)
core_cfg = EndpointCoreConfigInput(name=endpoint_name, served_entities=[served_entity])

try:
    w.serving_endpoints.get(name=endpoint_name)
    w.serving_endpoints.update_config(name=endpoint_name, served_entities=[served_entity])
    print(f"Updated endpoint '{endpoint_name}' to version {model_version}")
except NotFound:
    w.serving_endpoints.create(name=endpoint_name, config=core_cfg)
    print(f"Created endpoint '{endpoint_name}' (version {model_version})")


### Validate Deployment

After deployment:

- Confirm endpoint status is `READY`
- Test with sample inference request
- Verify output schema matches expected signature

Avoid exposing endpoints before validation.

In [ ]:
# Wait for endpoint to be READY using the SDK's built-in waiter
# (avoids manual enum comparison which silently fails)
import datetime
from databricks.sdk.service.serving import EndpointStateReady

ep = w.serving_endpoints.wait_get_serving_endpoint_not_updating(
    endpoint_name,
    timeout=datetime.timedelta(minutes=10)
)
print(f"Endpoint '{endpoint_name}' is READY")

# Tag model version and set task values
client.set_model_version_tag(registered_model_name, str(model_version), "deployment_status", "deployed")
client.set_model_version_tag(registered_model_name, str(model_version), "deployment_timestamp", datetime.datetime.utcnow().isoformat())
client.set_model_version_tag(registered_model_name, str(model_version), "endpoint_name", endpoint_name)

dbutils.jobs.taskValues.set(key="model_name", value=registered_model_name)
dbutils.jobs.taskValues.set(key="model_version", value=str(model_version))
dbutils.jobs.taskValues.set(key="endpoint_name", value=endpoint_name)
dbutils.jobs.taskValues.set(key="deployment_status", value="deployed")

print(f"Deployment completed: {registered_model_name} v{model_version} -> {endpoint_name}")

### Enable Inference Logging

Inference logging writes every request and response to a Delta table in Unity Catalog.
This is a prerequisite for Lakehouse Monitoring — without it there is nothing to monitor.

Logged table: `{catalog}.{schema}.{endpoint_name}_payload`

Once enabled, attach a Lakehouse Monitor to that table via the Catalog UI or the SDK
to track prediction drift, data drift, and model performance over time.

In [0]:
from databricks.sdk.service.serving import (
    AutoCaptureConfigInput,
    EndpointCoreConfigInput,
    ServedEntityInput,
)

# Enable inference logging — writes request/response payloads to a Delta table
# Required for Lakehouse Monitoring on the endpoint
auto_capture = AutoCaptureConfigInput(
    catalog_name=catalog,
    schema_name=schema,
    enabled=True,
    table_name_prefix=endpoint_name,
)

w.serving_endpoints.update_config(
    name=endpoint_name,
    served_entities=[
        ServedEntityInput(
            name=f"{model_name}-v{model_version}",
            entity_name=registered_model_name,
            entity_version=model_version,
            workload_size=workload_size,
            scale_to_zero_enabled=scale_to_zero,
        )
    ],
    auto_capture_config=auto_capture,
)

print(f"Inference logging enabled.")
print(f"Payloads will be written to: {catalog}.{schema}.{endpoint_name}_payload")
print(f"Attach a Lakehouse Monitor to that table in the Catalog UI to track drift.")